In [0]:
import pyspark.sql.functions as f
import datetime as d



# Criação de parametros para ingestão dos arquivos 
dbutils.widgets.text("caminho_arquivo", "/Volumes/zeta_project/bronze/arquivos/", "1. Caminho Origem")
dbutils.widgets.text("data_ref", "", "Data (Deixe vazio para Ontem)")
dbutils.widgets.text("nome_arquivo", "", "Nome do arquivo")
dbutils.widgets.text("tabela_bronze", "zeta_project.bronze.tabela", "3. Nome da Tabela Bronze")
dbutils.widgets.dropdown("flag_limpeza", "1", ["0", "1"], "Executar limpeza? (0=Não, 1=Sim)")
dbutils.widgets.dropdown("tipo_arquivo", "csv", ["csv", "parquet", "json"], "Tipo de arquivo")


# Atribuição dos parametros
caminho_arquivo = dbutils.widgets.get("caminho_arquivo")
tabela_bronze = dbutils.widgets.get("tabela_bronze")
valor_reproc = dbutils.widgets.get("flag_limpeza")
valor_int_reproc = int(valor_reproc)
reproc = bool(valor_int_reproc)
tipo_arquivo = dbutils.widgets.get("tipo_arquivo")
data_ingestao = dbutils.widgets.get("data_ref")
nome_arquivo = dbutils.widgets.get("nome_arquivo")


# Validacao de recebimento de data(Utilizado para processamento de arquivos antigos)
if not data_ingestao:
    data_ref = (d.datetime.now() - d.timedelta(days=1)).strftime('%Y-%m-%d')
else:
    data_ref = data_ingestao


caminho_final = f"{caminho_arquivo}/ingest_date-{data_ref}/{nome_arquivo}"

# Verifica tipo de arquivo para ingerir:

if tipo_arquivo == "csv":
    df_ingest = spark.read.format("csv") \
    .option("Header", True) \
    .option("inferSchema", True) \
    .load(caminho_final) \
    .withColumn("arq_data_ingest", f.lit(data_ingestao)) \
    .withColumn("data_carga", f.current_timestamp())

elif tipo_arquivo == "parquet":
    df_ingest = spark.read.format("parquet") \
    .option("inferSchema", True) \
    .load(caminho_final) \
    .withColumn("arq_data_ingest", f.lit(data_ingestao)) \
    .withColumn("data_carga", f.current_timestamp())

elif tipo_arquivo == "json":
    df_ingest = spark.read.format("json") \
        .option("multiline", True) \
        .load(caminho_final) \
        .withColumn("arq_data_ingest", f.lit(data_ingestao)) \
        .withColumn("data_carga", f.current_timestamp())


# Verifica flag de reprocessamento e substitui apenas o arquivo a ser reprocessado

if reproc:
       df_ingest.write.format("delta") \
       .option("mergeSchema", "true") \
       .option("replaceWhere", f"arq_data_ingest = '{data_ref}'") \
       .mode("overwrite").saveAsTable(tabela_bronze)    
else:

    tabela_existe = spark.catalog.tableExists(tabela_bronze)

    if tabela_existe:
        df_ingest.write.format("delta") \
        .mode("append") \
        .saveAsTable(tabela_bronze)
    else:
        df_ingest.write.format("delta") \
        .mode("overwrite") \
        .clusterBy("arq_data_ingest") \
        .saveAsTable(tabela_bronze)    



